# FieldWave Tutorial 2
## Gradient and Optimization with [JAX](https://jax.readthedocs.io) and [Optax](https://optax.readthedocs.io/)

In [1]:
import functools

import optax

import jax.numpy as jnp

import fieldwave

In [2]:
backend = 1  # JAX

# common
psi = jnp.pi / 2  # pure s (TE-like); psi is the linear angle

n_top = 1  # n_topncidence
n_bot = 1  # n_transmission

theta = 0 * jnp.pi / 180  # angle of incidence
phi = 0 * jnp.pi / 180  # angle of rotation

wavelength = 900

thickness = [500., 1000.]  # thickness of each layer, from top to bottom.
period = [1000.]  # length of the unit cell. Here it's 1D.

fto = [10]

type_complex = jnp.complex128

In [3]:
ucell_1d_m = jnp.array([
    [[0, 0, 0, 1, 1, 1, 1, 0, 0, 0, ]],
    [[1, 1, 1, 1, 0, 1, 1, 1, 1, 1, ]],
    ]) * 4. + 1.  # refractive index

## 2.1 Gradient

Gradient can be calculated with the help of `jax.value_and_grad` function.
Read this for further information: [Automatic Differentiation](https://jax.readthedocs.io/en/latest/jax-101/04-advanced-autodiff.html)

Optax is used for optimization. Like PyTorch, Optax also provides various loss functions and optimizers so users easily can utilize well-established implementations. Refer this tutorial: [Learn Optax](https://optax.readthedocs.io/en/latest/optax-101.html)


Gradient can be utilized to solve optimization problems. Here are examples that show couple of ways to get gradient or optimized values with or without predefined functions of fieldwave.

### 2.1.1 Examples

In [4]:
class Loss:
    def __call__(self, result, *args, **kwargs):
        res_psi, res_te, res_ti = result.res, result.res_te_inc, result.res_tm_inc
        T = res_psi.T
        center = [a // 2 for a in T.shape]
        res = T[center[0], center[1]+1]

        return res


loss_fn = Loss()

In [5]:
solver = fieldwave.rcwa(backend=backend, psi=psi, n_top=n_top, n_bot=n_bot, theta=theta, phi=phi, fto=fto, wavelength=wavelength, period=period, ucell=ucell_1d_m, thickness=thickness, type_complex=type_complex)

pois = ['ucell', 'thickness']
# Freeze the discrete geometry decisions once, so jax.jit has nothing left to
# trace through. This sampled path no longer needs it, but NVF and the integrated path do.
geometry = solver.freeze_geometry()
forward = functools.partial(solver.conv_solve, geometry=geometry)

# case 1: Gradient
grad = solver.grad(pois, forward, loss_fn)

print('ucell gradient:')
print(grad['ucell'])
print('thickness gradient:')
print(grad['thickness'])

ucell gradient:
[[[-0.05138452 -0.02542783 -0.00732126  0.07866144 -0.01875361
    0.09488402  0.08794969 -0.00126606 -0.03663523 -0.04809363]]

 [[-0.17826879 -0.0829977  -0.22139839 -0.19180767  0.09207949
    0.05883863 -0.04205759 -0.13477581 -0.29789619  0.13094639]]]
thickness gradient:
[ 0.00222953 -0.00675975]


## 2.2 Optimization

### 2.2.1 Examples

Example 1

In [6]:
solver = fieldwave.rcwa(backend=backend, psi=psi, n_top=n_top, n_bot=n_bot, theta=theta, phi=phi, fto=fto, wavelength=wavelength, period=period, ucell=ucell_1d_m, thickness=thickness, type_complex=type_complex)

pois = ['ucell', 'thickness']
# Freeze the discrete geometry decisions once, so jax.jit has nothing left to
# trace through. This sampled path no longer needs it, but NVF and integrated do.
geometry = solver.freeze_geometry()
forward = functools.partial(solver.conv_solve, geometry=geometry)

# case 2: SGD
optimizer = optax.sgd(learning_rate=1e-2, momentum=0.9)
res = solver.fit(pois, forward, loss_fn, optimizer, iteration=3)

print('ucell final:')
print(res['ucell'])
print('thickness final:')
print(res['thickness'])

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:00<00:01,  1.23it/s]

100%|██████████| 3/3 [00:00<00:00,  3.68it/s]

ucell final:
[[[1.00287801 1.00146043 1.00050035 4.99665374 5.00175393 4.99576938
   4.99614966 1.00015046 1.00215507 1.00276536]]

 [[5.00541901 4.99982157 5.00824523 5.0064494  0.99316574 4.99243601
   4.99821826 5.00365639 5.01333214 4.9885417 ]]]
thickness final:
[ 499.9998919  1000.00039653]
